# 🎓 Fine-Tuning GPT-2 on Custom Text Data
### Internship Task — GCEK CodeBreakers Club

**Objective:** Fine-tune GPT-2 (a transformer-based language model by OpenAI) on a custom dataset so it generates coherent, contextually relevant text that mimics the style and structure of your training data.

**What you will learn:**
- How to prepare a custom Q&A dataset for language model training
- How to fine-tune GPT-2 using Hugging Face `transformers`
- How to generate text using the fine-tuned model

**References:** [Hugging Face Docs](https://huggingface.co/docs/transformers) | [GPT-2 Paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

---
> ⚠️ **Before you start:** Go to **Runtime → Change runtime type → T4 GPU** to enable GPU acceleration.

## Step 1 — Verify GPU

In [1]:
# Confirm a GPU is available
!nvidia-smi

Sun Apr 19 17:53:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2 — Install Dependencies

In [2]:
# Install required libraries
!pip install -q transformers datasets accelerate
print("Libraries installed successfully!")

Libraries installed successfully!


## Step 3 — Prepare the Training Dataset

We format data as Q&A pairs wrapped with special tokens `<|startoftext|>` and `<|endoftext|>`.  
This teaches GPT-2 the pattern: **Question → Answer**.

> 💡 **Tip for Interns:** You can expand `add_pair(...)` with more Q&A pairs about your own college, project, or any domain. The more diverse and high-quality the data, the better the model!

In [14]:
DATASET_FILE = "gcek_dataset.txt"   # Output file name

## Step 4 — Load GPT-2 Model & Tokenizer

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2"   # Options: "gpt2", "gpt2-medium", "EleutherAI/gpt-neo-125M"

print(f"Loading model: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# GPT-2 doesn't have a pad token by default; use EOS token as pad
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model loaded! Total parameters: {total_params:.1f}M")

Loading model: gpt2 ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded! Total parameters: 124.4M


## Step 5 — Tokenize & Prepare Dataset

In [16]:
from datasets import load_dataset

# Load the text dataset
dataset = load_dataset("text", data_files=DATASET_FILE, split="train")
# Filter empty lines
dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)
print(f"Dataset loaded: {len(dataset)} lines")

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# Group tokens into fixed-size blocks for language modeling
BLOCK_SIZE = 128

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = (len(concatenated["input_ids"]) // BLOCK_SIZE) * BLOCK_SIZE
    result = {
        k: [t[i : i + BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset = tokenized_dataset.map(group_texts, batched=True)
print(f"Tokenization complete. Total blocks: {len(lm_dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/259 [00:00<?, ? examples/s]

Dataset loaded: 208 lines


Map:   0%|          | 0/208 [00:00<?, ? examples/s]

Map:   0%|          | 0/208 [00:00<?, ? examples/s]

Tokenization complete. Total blocks: 26


## Step 6 — Configure Training

**Key hyperparameters explained:**
| Parameter | Value | Meaning |
|---|---|---|
| `num_train_epochs` | 15 | How many full passes over the dataset |
| `learning_rate` | 5e-5 | How fast the model updates weights |
| `per_device_train_batch_size` | 4 | Samples processed per GPU step |
| `weight_decay` | 0.01 | Regularization to prevent overfitting |

In [18]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

OUTPUT_DIR = "./gcek_gpt2_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=100,   # was 15 — need much more on tiny data
    learning_rate=1e-4,     # slightly higher to force memorization
    per_device_train_batch_size=2,  # smaller batch on tiny dataset
    # learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,          # Keep only last 2 checkpoints to save disk space
    prediction_loss_only=True,
    fp16=True,                   # Mixed precision — faster on T4/V100 GPU
    report_to="none",            # Disable W&B logging
)

# Data collator handles dynamic padding
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False    # GPT-2 is causal LM, not masked LM
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset,
    data_collator=data_collator,
)

print("Trainer configured and ready!")

Trainer configured and ready!


## Step 7 — Train the Model

> ⏳ Training takes approximately **3–6 minutes** on a T4 GPU with this dataset size.  
> Watch the **loss** value — it should decrease over time. Lower loss = better model.

In [19]:
print("Starting training...\n")
trainer.train()
trainer.state.log_history[-1]
print("\nTraining complete!")

Starting training...



Step,Training Loss
10,3.933965
20,3.127452
30,2.604591
40,2.179672
50,1.661550
60,1.371156
70,0.945843
80,0.796682
90,0.619916
100,0.465960


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete!


## Step 8 — Save the Fine-Tuned Model

In [20]:
SAVE_DIR = "./gcek_gpt2_finetuned"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model saved to '{SAVE_DIR}'")
!ls -lh {SAVE_DIR}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to './gcek_gpt2_finetuned'
total 479M
-rw-r--r-- 1 root root  962 Apr 19 18:10 config.json
-rw-r--r-- 1 root root  118 Apr 19 18:10 generation_config.json
-rw-r--r-- 1 root root 475M Apr 19 18:10 model.safetensors
-rw-r--r-- 1 root root  297 Apr 19 18:10 tokenizer_config.json
-rw-r--r-- 1 root root 3.4M Apr 19 18:10 tokenizer.json


## Step 9 — Generate Text from Fine-Tuned Model

Try different prompts and observe how the model generates relevant text!

In [21]:
from transformers import pipeline

# Load the saved model into a text-generation pipeline
generator = pipeline(
    "text-generation",
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0   # 0 = GPU, -1 = CPU
)

print("Generator pipeline ready!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Generator pipeline ready!


In [22]:
# ─────────────────────────────────────────────────────────────────
#   CHANGE THE PROMPT BELOW AND RE-RUN THIS CELL TO EXPERIMENT!
# ─────────────────────────────────────────────────────────────────

prompts_to_test = [
    "Question: What is GCEK?\nAnswer:",
    "Question: What courses are offered at GCEK?\nAnswer:",
    "Question: Tell me about CodeBreakers.\nAnswer:",
]

for prompt in prompts_to_test:
    print("=" * 60)
    print(f"PROMPT: {prompt.strip()}")
    outputs = generator(
        prompt,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.6,       # Lower = more focused output
        top_k=40,              # Limits to top 40 token choices
        top_p=0.9,             # Nucleus sampling
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = outputs[0]["generated_text"].replace(prompt, "").strip()
    print(f"GENERATED: {generated}")
    print()

Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: Question: What is GCEK?
Answer:


Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED: Government College of Engineering Kalahandi (GCEk) is a public autonomous engineering college located in Bhawanipatna, Odisha, India. It is affiliated with Biju Patnaik University of Technology (BPUT), became an active constituent university on 21 January 2021. The boys hostel is Visvesvaraya Hall of Residency and APJ Hall of Excellence. Each department

PROMPT: Question: What courses are offered at GCEK?
Answer:


Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED: Government College of Engineering Kalahandi (GCEk) offers B.Tech programs in four branches — Civil Engineers, Computer Science Engineering (CSE), Electrical Engineer and Mechanical engineer—in total., MLRS Pvt. Ltd. (MoU), Etricals Electromania Limited Partnership Co. Odisha (ETRANO), Rialto Brasileiro Seoczar (S

PROMPT: Question: Tell me about CodeBreakers.
Answer:
GENERATED: What is the website of Codebreakersgcek?Answer, www2codebreakerscse.tech. No prior coding experience is required to join —the club welcomes beginners and helps them build skills from the ground up. The boys hostel is AWAHAAN University of Technology (BPUT), accessible at awahaan.ac., or emailed at info@awspatnaurcan



## Step 10 — Experiment & Reflect

Use the cell below to freely test your model with any prompt.

In [26]:
my_prompt = "Question: What is GCEK?\nAnswer:"

outputs = generator(
    my_prompt,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.3,
    num_return_sequences=3,
    pad_token_id=tokenizer.eos_token_id,
)

print(f"Prompt: {my_prompt}\n")
for i, out in enumerate(outputs, 1):
    generated = out["generated_text"].replace(my_prompt, "").strip()
    print(f"[Output {i}]: {generated}\n")

Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: Question: What is GCEK?
Answer:

[Output 1]: Government College of Engineering Kalahandi (GCEk) is a public autonomous engineering college located in Bhawanipatna, beside NH-26J road and across the river from Junagarh. It is affiliated with Biju Patnaik University of Technology (BPUT), Odisha., became an active constituent university of AICTE on 21 January 2021. The boys hostel is Mother Teresa Hall of Residency at APWODL. These two universities are also open to all

[Output 2]: Government College of Engineering Kalahandi (GCEk) is a public autonomous engineering college located in Bhawanipatna, Rajas Medical Campus at Bandopala, India. It is affiliated with Biju Patnaik University of Technology (BPUT), Odisha., and became an independent university on 21 January 2021. The boys hostel is Mother Teresa Hall of Residency — an authentic space for students to socialize together, learn from each other through music, dance classes, drama

[Output 3]: Government College of Engineering 

---
## Summary

| Step | What happened |
|------|---------------|
| Dataset | Created Q&A pairs formatted with GPT-2 special tokens |
| Model | Loaded pre-trained GPT-2 (124M parameters) from Hugging Face |
| Tokenization | Converted text to token IDs in fixed-size blocks |
| Training | Fine-tuned GPT-2 on GCEK data for 15 epochs |
| Inference | Generated contextual answers from a given prompt |

## 💡 Ideas to Extend This Project
- Add **50+ more Q&A pairs** on different topics for richer generation
- Try **`gpt2-medium`** (345M params) for better quality (needs more VRAM)
- Try **`EleutherAI/gpt-neo-125M`** as an open-source alternative
- Build a simple **Gradio UI** to let users interact with the model
- Evaluate output quality using **BLEU score** or **perplexity**

---
*Notebook prepared for GCEK Internship — CodeBreakers Club*